In [11]:
from google.colab import drive
drive.mount("/content/gdrive/", force_remount=True)
import numpy as np
import pandas as pd
import math
import os
import time
from datetime import timedelta

Mounted at /content/gdrive/


In [12]:
#Hàm trả về giá trị rating dự đoán của user trên item i bằng cách sử dụng phương pháp chọn láng giềng BS
#BS - Baseline Strategy: Lấy top K láng giềng có rate trên item i
#Input: user, i:item, k: số lượng láng giềng
#Output: rating dự đoán của user trên i

def BaselinePrediction(user, i, training_df, user_average_rating_df, sim_path, k):
  user_df = pd.read_excel(r"{0}/User{1}.xlsx".format(sim_path, user))

  #lấy ra những user có rate trên item i
  user_rating_i = training_df.User[training_df.Item == i].to_list()

  #lấy top k user
  user_top_k = user_df.User[user_df.User.isin(user_rating_i) == True][:k].to_list()
  alpha = 0
  beta = 0

  for j in user_top_k:
    #Tính toán
    R_j_tb = user_average_rating_df.UserAvg[user_average_rating_df.User == j].values[0]
    user_j = training_df.Rating[training_df.User == j][training_df.Item == i].values[0]
    sim_x_y = user_df.Sim[user_df.User == j].values[0]
    alpha = alpha + (user_j - R_j_tb)*sim_x_y
    beta = beta + abs(sim_x_y)
    
  if beta == 0:
    return user_average_rating_df.UserAvg[user_average_rating_df.User == user].values[0]
  return (user_average_rating_df.UserAvg[user_average_rating_df.User == user].values[0] + (alpha/beta))


In [13]:
#Hàm trả về danh sách những láng giềng được lấy theo phương pháp SS
#SS: Top k láng giềng theo sim cao nhất (không quan tâm các điều kiện ràng buộc khác)
#Input: user, i:item, k: số lượng láng giềng, active_user: user chính trong thuật toán RPA
#Output: danh sách láng giềng của user trên item i

def SelectNeighbor(user, i, training_df, sim_path, k, active_user):
  #Đọc file độ tương đồng của user
  user_df = pd.read_excel(r"{0}/User{1}.xlsx".format(sim_path, user))

  #Lấy top k user đầu tiên
  list_user_SS = user_df.User[:k].to_list()

  #Kiểm tra trường hợp nếu trùng với active user
  if active_user in list_user_SS:
    list_user_SS.remove(active_user)
    list_user_SS.append(user_df.User[k:k+1].values[0])
    
  return list_user_SS


In [14]:
#Input: user, i:item, level: mức độ level đệ quy hiện tại, k: số lượng láng giềng
#Output: rating dự đoán của user trên item i

def RecursivePrediction(user, i, training_df, user_average_rating_df, level, sim_path, k, active_user, level_counts, level_rated_neighbor):
  #Đọc file độ tương đồng
  user_df = pd.read_excel(r"{0}/User{1}.xlsx".format(sim_path, user))
  level_rated_neighbor = level

  #Kiểm tra level hiện tại
  if level >=2:
    return BaselinePrediction(user, i, training_df, user_average_rating_df, sim_path, k), level_counts, level_rated_neighbor
  
  #Danh sách láng giềng của user
  list_U = SelectNeighbor(user, i, training_df, sim_path, k, active_user)

  alpha = 0
  beta = 0

  #Duyệt qua từng láng giềng
  for j in list_U:
    if level == 0:
      level_rated_neighbor = 0

    #Kiểm tra nếu láng giềng có đánh giá trên item
    if not (training_df.Rating[training_df.User == j][training_df.Item == i].empty):
      alpha = alpha + ((int(training_df.Rating[training_df.User == j][training_df.Item == i].values[0]  *10 ** decimal_places) /10 ** decimal_places)
      - user_average_rating_df.UserAvg[user_average_rating_df.User == j].values[0]) * user_df.Sim[user_df.User == j].values[0]
      beta = beta + abs(user_df.Sim[user_df.User == j].values[0])
      
    #Nếu láng giềng không đánh giá thì gọi đệ quy dự đoán
    else:
      rating_j, level_counts, level_rated_neighbor = RecursivePrediction(j, i, training_df, user_average_rating_df, level + 1, sim_path, k, active_user, level_counts, level_rated_neighbor)
      
      #Thực hiện phép tính sau khi thoát đệ quy
      alpha = alpha + 0.5 * ((int(rating_j  *10 ** decimal_places) /10 ** decimal_places)
      - user_average_rating_df.UserAvg[user_average_rating_df.User == j].values[0]) * user_df.Sim[user_df.User == j].values[0]
      beta = beta + 0.5 * abs(user_df.Sim[user_df.User == j].values[0])
      
    if level == 0:
      level_counts[level_rated_neighbor] += 1
  if beta == 0:
    return user_average_rating_df.UserAvg[user_average_rating_df.User == user].values[0], level_counts, level_rated_neighbor
  return ( user_average_rating_df.UserAvg[user_average_rating_df.User == user].values[0] + (alpha/beta)), level_counts, level_rated_neighbor


In [15]:
def recommend_list(user, N, k, training_df, testing_df, user_average_rating_df, real_pre_DataFrame, sim_path):
  user_col = []
  user_col_rmse=[]
  item_col=[]
  rating_real_col=[]
  rating_predict_col=[]
  level_col=[]
  level0_col=[]
  level1_col=[]
  level2_col=[]

  #Danh sách các item cần dự đoán
  predict_list = testing_df.Item[testing_df.User == user].to_list()
  #vòng lặp item
  for j in predict_list:
    level_counts = [0, 0, 0]
    level_rated_neighbor = 0
    rpa_rating, level_counts, level_rated_neighbor = RecursivePrediction(user, j, training_df, user_average_rating_df, 0, sim_path, k, user, level_counts, level_rated_neighbor)
    
    #Tìm rating trong test
    rate_real =  testing_df.Rating[testing_df.User == user][testing_df.Item == j].values[0]
    #thêm id item j vào cột item
    user_col.append(user)
    item_col.append(j)
    #Thêm rating vào các cột
    rating_real_col.append(rate_real)
    rating_predict_col.append(rpa_rating)
    user_col_rmse.append(np.sqrt(pow(rpa_rating - rate_real,2)))
    if level_counts[2] != 0:
      level_col.append(2)
    else:
      if level_counts[1] != 0:
        level_col.append(1)
      else:
        level_col.append(0)
    level0_col.append(level_counts[0])
    level1_col.append(level_counts[1])
    level2_col.append(level_counts[2])

  df = pd.DataFrame(data={'User':user_col,'Item':item_col,'Rating_real':rating_real_col,'Rating_predict':rating_predict_col, 'RMSE': user_col_rmse, 'Level_rated': level_col, 'Level0': level0_col, 'Level1': level1_col, 'Level2': level2_col})
  #Sắp xếp giảm dần theo rating dự đoán
  df = df.sort_values(by=['Rating_predict'],axis=0,ascending=False, ignore_index=True)
  real_pre_DataFrame = pd.concat([real_pre_DataFrame, df])

  #Lấy danh sách item từ 1 tới N
  df =  df.iloc[:N]

  #Trả về danh sách N item
  return df.Item.to_list(), real_pre_DataFrame

In [16]:
def revaluation(N, k, sim_path, testing_df, training_df, first, last):
  #Khai báo trước những thước đo đánh giá thành các cột
  real_pre_df = pd.DataFrame()
  User_col = []
  I_N_col = []
  I_te_col = []
  intersection_col = []
  I_N_len_col = []
  I_te_len_col = []
  intersection_len_col = []
  Prec_u_N = []
  Rec_u_N = []
  F1_u_N = []

  testing_user = testing_df.User[testing_df.User>=0].to_list()
  testing_user = list(dict.fromkeys(testing_user))
  #Vòng lặp người dùng
  for i in range(first, last + 1):
    #if i in testing_user :
    if i in testing_user and i in training_df.User.to_list():
      #Danh sách item system recommend cho người dùng
      I_N, real_pre_df = recommend_list(i, N, k, training_df, testing_df, user_avg_df, real_pre_df, sim_path)

      #Danh sách item trong file test của người dùng
      I_te = testing_df.Item[testing_df.User == i][testing_df.Rating > 3 ].to_list()

      #Phần giao giữa danh sách item
      intersection = [value for value in I_N if value in I_te]
      try:
        #Tính thước đo đánh giá
        Prec =  abs(len(intersection)) / abs(len(I_N))
        Rec = abs(len(intersection)) / abs(len(I_te))
        F1 = 2 * (Prec * Rec) / (Prec + Rec)
      except:
        #Trường hợp các list này rỗng
        if intersection == [] or I_te == [] or I_N == []:
          Prec = 0
          Rec = 0
          F1 = 0
    else:
      continue
    #Thêm id người dùng, các độ đo vào các cột
    User_col.append(i)
    I_N_col.append(' '.join(str(e)for e in I_N))
    I_te_col.append(' '.join(str(e)for e in I_te))
    intersection_col.append(' '.join(str(e)for e in intersection))
    I_N_len_col.append(len(I_N))
    I_te_len_col.append(len(I_te))
    intersection_len_col.append(len(intersection))
    Prec_u_N.append(Prec)
    Rec_u_N.append(Rec)
    F1_u_N.append(F1)

  #Tạo dataframe
  df = pd.DataFrame(data={'User':User_col, 'Tập tư vấn (C)':I_N_col, 'Tập đúng (T)':I_te_col, 'Giao TC':intersection_col, 'C_length':I_N_len_col, 'T_length':I_te_len_col, 'TC_length':intersection_len_col, 'Precision':Prec_u_N, 'Recall': Rec_u_N, 'F1': F1_u_N})
  #Tính tổng F1
  if df.C_length.sum() > 0:
    pre = df.TC_length.sum()/df.C_length.sum()
  else:
      pre = 0

  if df.T_length.sum() > 0:
      recall = df.TC_length.sum()/df.T_length.sum()
  else:
      recall = 0
  if pre + recall > 0:
    f1 = 2 * pre * recall / (pre + recall)
  else:
    f1 = 0

  df_all = pd.DataFrame(data={'Precision_system':[pre], 'Recall_system':[recall], 'F1':[f1]})
  #Tính tổng RMSE
  all = np.sqrt( real_pre_df.RMSE.sum()/real_pre_df.RMSE.notnull().sum() )
  real_pre_df_all = pd.DataFrame(data={'RMSE_system':[all]})

  # Xuất file kết quả
  result_path = os.path.join(sim_path[0:-7], 'result_RPA_SS_detail')
  if not os.path.exists(result_path):
      os.makedirs(result_path)

  # Chỉ cần để ý F1 score và RMSE
  file_path = os.path.join(result_path, "{}_N{}_k{}_RPA.xlsx".format(name, N, k))
  with pd.ExcelWriter(file_path) as writer:
    df.to_excel(writer, sheet_name = 'F1_RMSE', index = False) #F1
    df_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 10, index = False) #F1 tổng
    real_pre_df.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 13, index = False) #RMSE
    real_pre_df_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 22, index = False) #RMSE tổng

In [17]:
def convert_seconds_to_time(seconds):
    # Chuyển số giây thành đối tượng timedelta
    duration = timedelta(seconds=seconds)

    # Tính toán các thành phần ngày, giờ, phút và giây
    days = duration.days
    hours, remainder = divmod(duration.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    # Tạo chuỗi định dạng ngày:giờ:phút:giây
    formatted_time = "{:02}d:{:02}h:{:02}m:{:02}s".format(days, hours, minutes, seconds)

    return formatted_time

In [ ]:
#RUN
name = 'MovieLen100k'


decimal_places = 9
#đọc file training, testing
training_df = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/{0}_training.xlsx'.format(name), dtype={'User': int, 'Item': int})
testing_df = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/{0}_testing.xlsx'.format(name), dtype={'User': int, 'Item': int})
#Đọc file User_Average
user_avg_df = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/User_Average.xlsx'.format(name))
user_avg_df = user_avg_df.rename(columns={'User_Average': 'UserAvg'}, errors="raise")
#số lượng user
n_user = np.max(training_df.User)
N = 5
k = 20

#đo thời gian
start_time = time.time()
revaluation(N, k,"/content/gdrive/MyDrive/Luận văn/{0}/SimUser".format(name), testing_df, training_df, 97, 97)
end_time = time.time()

#đọc file kết quả
data = pd.read_excel(r'/content/gdrive/MyDrive/Luận văn/{0}/result_RPA_SS_detail/{0}_N{1}_k{2}_RPA.xlsx'.format(name, N, k))

#tạo dữ liệu cho time
time_all = pd.DataFrame({'Time_all': [convert_seconds_to_time(end_time - start_time)]})

#Thêm cột time vào file excel
file_path = os.path.join(r'/content/gdrive/MyDrive/Luận văn/{0}/result_RPA_SS_detail'.format(name), "{0}_N{1}_k{2}_RPA.xlsx".format(name, N, k))
with pd.ExcelWriter(file_path) as writer:
  data.to_excel(writer, sheet_name = 'F1_RMSE', index = False)
  time_all.to_excel(writer, sheet_name = 'F1_RMSE', startcol = 23, index = False)
